<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
from sklearn.model_selection import train_test_split


def load_data(seed, validation_size=0.2, sample_size=None):
    """Carrega, achata, normaliza e separa o CIFAR-10 em treino/valida??o.

    sample_size ? opcional e ajuda a executar os experimentos mais r?pido em
    computadores modestos. Para usar o treino completo, deixe sample_size=None.
    """
    from tensorflow.keras.datasets import cifar10

    (X_train_full, y_train_full), _ = cifar10.load_data()

    y_train_full = y_train_full.ravel()
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1).astype("float32") / 255.0

    if sample_size is not None and sample_size < X_train_full.shape[0]:
        X_train_full, _, y_train_full, _ = train_test_split(
            X_train_full,
            y_train_full,
            train_size=sample_size,
            random_state=seed,
            stratify=y_train_full,
        )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size,
        random_state=seed,
        stratify=y_train_full,
    )

    return X_train, X_val, y_train, y_val


SEED = 42
X_train, X_val, y_train, y_val = load_data(SEED, sample_size=10000)

print("Treino:", X_train.shape, y_train.shape)
print("Valida??o:", X_val.shape, y_val.shape)
print("Menor/Maior valor normalizado:", X_train.min(), X_train.max())

**Respostas da Quest?o 1**

1. O formato original das imagens do CIFAR-10 ? `(32, 32, 3)`: altura 32, largura 32 e tr?s canais RGB.
2. Ap?s o flatten, cada imagem possui `32 * 32 * 3 = 3072` features.
3. O flatten ? necess?rio porque uma MLP recebe vetores unidimensionais na camada de entrada, n?o matrizes/imagens com estrutura espacial expl?cita.
4. A normaliza??o coloca os pixels na escala `[0, 1]`, o que melhora a estabilidade num?rica, evita que gradientes fiquem muito grandes e facilita a converg?ncia do otimizador.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=30,
    batch_size=128,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=max_iter,
        batch_size=batch_size,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        verbose=False,
    )

    model.fit(X_train, y_train)
    return model


# Exemplo r?pido de treino
example_model = train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=SEED,
)

print("?pocas treinadas:", example_model.n_iter_)
print("?ltima loss:", example_model.loss_)

**Respostas da Quest?o 2**

1. Na primeira camada, o n?mero de par?metros ? `(n_features * n_neur?nios) + n_neur?nios`. Para CIFAR-10 com 3072 features e uma camada com 64 neur?nios, isso resulta em `3072 * 64 + 64 = 196672` par?metros.
2. A ReLU retorna zero para entradas negativas e mant?m entradas positivas. Isso introduz n?o linearidade, reduz satura??o em compara??o com sigmoid/tanh e costuma acelerar o treinamento.
3. MLPs possuem muitos par?metros em imagens porque cada pixel/canal vira uma entrada conectada a todos os neur?nios da primeira camada. Como a imagem tem 3072 entradas, mesmo camadas pequenas j? geram centenas de milhares de pesos.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    return metrics


example_metrics = evaluate(example_model, X_val, y_val)
pd.DataFrame([example_metrics])

**Interpreta??o e respostas da Quest?o 3**

A accuracy indica a propor??o total de imagens classificadas corretamente. Precision mede, entre os exemplos previstos como uma classe, quantos realmente pertencem a ela; recall mede, entre os exemplos reais de uma classe, quantos o modelo conseguiu recuperar. O f1-score combina precision e recall e ? especialmente ?til quando h? desequil?brio entre classes ou quando falsos positivos e falsos negativos precisam ser avaliados em conjunto.

No CIFAR-10, uma MLP tende a ter desempenho limitado porque perde a estrutura espacial da imagem ao aplicar flatten. Ainda assim, as m?tricas s?o ?teis para comparar hiperpar?metros sob as mesmas condi??es experimentais.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import time


def run_experiment(
    run_name,
    X_train,
    y_train,
    X_val,
    y_val,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=42,
    max_iter=30,
    batch_size=128,
):
    params = {
        "activation": activation,
        "hidden_layers": str(hidden_layers),
        "learning_rate": learning_rate,
        "max_iter": max_iter,
        "batch_size": batch_size,
        "seed": seed,
    }

    with mlflow.start_run(run_name=run_name):
        for key, value in params.items():
            mlflow.log_param(key, value)

        start = time.time()
        model = train_mlp(
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size,
        )
        training_time = time.time() - start

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time
        metrics["n_iter"] = model.n_iter_
        metrics["final_loss"] = float(model.loss_)

        for key, value in metrics.items():
            mlflow.log_metric(key, value)

    return {**params, **metrics}, model


baseline_result, baseline_model = run_experiment(
    "baseline_relu_64_lr_001",
    X_train,
    y_train,
    X_val,
    y_val,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=SEED,
)

pd.DataFrame([baseline_result])

**Respostas da Quest?o 4**

O experimento com melhor desempenho deve ser identificado pela maior `accuracy` e pelo maior `f1_score` na valida??o. A configura??o mais est?vel ? aquela que combina boa m?trica com loss final menor, menos oscila??es e converg?ncia sem parar cedo por instabilidade. O principal benef?cio do MLflow ? centralizar par?metros e m?tricas, permitindo comparar execu??es de forma reprodut?vel em vez de depender de anota??es manuais.

No baseline acima, a configura??o `relu`, `(64,)`, `learning_rate=0.001` serve como refer?ncia para comparar ativa??es, arquiteturas e learning rates nas quest?es seguintes.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activation_results = []
activation_models = {}

for activation in ["logistic", "tanh", "relu"]:
    result, model = run_experiment(
        run_name=f"activation_{activation}",
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        seed=SEED,
    )
    activation_results.append(result)
    activation_models[activation] = model

activation_results_df = pd.DataFrame(activation_results).sort_values(
    by="f1_score",
    ascending=False,
)
activation_results_df

**Respostas da Quest?o 5**

Em geral, a ReLU apresenta melhor converg?ncia em MLPs porque sofre menos com satura??o de gradiente do que `logistic` e `tanh`. A maior estabilidade deve ser observada na ativa??o que mant?m boa accuracy, menor loss e n?mero de itera??es consistente; normalmente ReLU ou `tanh` tendem a se comportar melhor do que `logistic` nesse cen?rio.

As diferen?as podem ser significativas porque `logistic` satura com facilidade, `tanh` centraliza as ativa??es em torno de zero mas ainda pode saturar, e ReLU preserva gradientes positivos de forma mais direta. Por isso, ReLU ? amplamente utilizada em Deep Learning: ela ? simples, eficiente, reduz problemas de gradiente desaparecendo e costuma acelerar o aprendizado em redes profundas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architecture_results = []
architecture_models = {}

for hidden_layers in [(32,), (64,), (128, 64), (256, 128)]:
    result, model = run_experiment(
        run_name=f"architecture_{hidden_layers}",
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        seed=SEED,
    )
    architecture_results.append(result)
    architecture_models[hidden_layers] = model

architecture_results_df = pd.DataFrame(architecture_results).sort_values(
    by="f1_score",
    ascending=False,
)
architecture_results_df

**Respostas da Quest?o 6**

Redes maiores n?o necessariamente melhoram os resultados. Elas aumentam a capacidade do modelo, mas tamb?m elevam o custo computacional e o risco de overfitting. O melhor tradeoff costuma ser a menor arquitetura que alcan?a m?tricas pr?ximas da melhor, com tempo de treinamento menor e loss est?vel.

Sinais de overfitting aparecem quando a rede melhora muito no treino, mas n?o melhora na valida??o, ou quando a loss de valida??o piora. A arquitetura `(256, 128)` tende a ter maior custo computacional, pois possui mais camadas, mais neur?nios e mais par?metros conectados ?s 3072 features de entrada.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rate_results = []
learning_rate_models = {}

for learning_rate in [0.1, 0.01, 0.001]:
    result, model = run_experiment(
        run_name=f"learning_rate_{learning_rate}",
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(64,),
        learning_rate=learning_rate,
        seed=SEED,
    )
    learning_rate_results.append(result)
    learning_rate_models[learning_rate] = model

learning_rate_results_df = pd.DataFrame(learning_rate_results).sort_values(
    by="f1_score",
    ascending=False,
)
learning_rate_results_df

**Respostas da Quest?o 7**

O melhor learning rate ? o que obt?m maior `f1_score`/accuracy com loss final menor e treinamento est?vel. Em MLPs para CIFAR-10, `0.001` ou `0.01` normalmente s?o mais seguros que `0.1`. O valor `0.1` tende a ser o mais inst?vel, pois passos grandes podem ultrapassar regi?es boas da fun??o de perda e causar oscila??es ou m? converg?ncia.

Quando o learning rate ? muito alto, a loss pode oscilar ou divergir. Quando ? muito baixo, o treinamento fica lento e pode parar antes de encontrar uma solu??o competitiva dentro do n?mero de ?pocas definido.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
all_results = pd.concat(
    [
        pd.DataFrame([baseline_result]),
        activation_results_df,
        architecture_results_df,
        learning_rate_results_df,
    ],
    ignore_index=True,
).drop_duplicates()

best_result = all_results.sort_values(by="f1_score", ascending=False).iloc[0]

print("Melhor configura??o observada:")
print(best_result[["activation", "hidden_layers", "learning_rate", "accuracy", "f1_score", "final_loss", "training_time"]])

all_results.sort_values(by="f1_score", ascending=False)

**Discuss?o final da Quest?o 8**

A loss representa o erro que a rede tenta minimizar por backpropagation. Nos experimentos, learning rates moderados tendem a produzir queda de loss mais est?vel; learning rates altos podem causar oscila??es, enquanto valores muito baixos tornam a converg?ncia lenta. A arquitetura influencia diretamente a capacidade do modelo: redes pequenas podem subajustar, enquanto redes grandes podem aumentar custo e overfitting sem garantir melhora proporcional.

As fun??es de ativa??o tamb?m afetam o treinamento. A ReLU costuma convergir melhor porque preserva gradientes em ativa??es positivas, enquanto `logistic` e `tanh` podem saturar. Mesmo com bons ajustes, a MLP tem limita??es para imagens porque o flatten remove rela??es espaciais locais, como bordas, texturas e vizinhan?a entre pixels. CNNs lidam melhor com esse tipo de dado justamente por explorar estrutura espacial.

A melhor configura??o final ? a que aparece em `best_result`, isto ?, a combina??o com maior `f1_score` na valida??o. As principais dificuldades observadas s?o o alto n?mero de par?metros, o custo computacional, a sensibilidade a hiperpar?metros e a limita??o estrutural da MLP para imagens. O backpropagation contribui calculando gradientes da loss em rela??o aos pesos, permitindo atualizar a rede iterativamente para reduzir o erro de classifica??o.